In [ ]:
# Text Similarity using Word Mover's Distance (WMD)

In [4]:
%%capture
!pip install gensim nltk POT scikit-learn --quiet

### What is Word Mover's Distance (WMD)?

Word Mover's Distance (WMD) is a measure of the dissimilarity between two text documents. Unlike traditional text similarity metrics that treat documents as bag-of-words, WMD leverages word embeddings (like Word2Vec) to understand the semantic meaning of words. It calculates the minimum 'distance' that the words in one document need to 'travel' to match the words in another document. This allows WMD to capture subtle semantic similarities, even if the documents don't share many common words.

### How WMD Works:

1.  **Word Embeddings**: Each word in the documents is represented as a vector in a high-dimensional space, where semantically similar words are close to each other.
2.  **Earth Mover's Distance**: WMD then uses a concept similar to Earth Mover's Distance (EMD). Imagine the words in one document as 'piles of dirt' and the words in the other document as 'holes' that need to be filled. WMD finds the minimum cumulative distance (cost) to move the 'dirt' (words) from the first document to cover the 'holes' (words) in the second document.
3.  **Optimization**: This is an optimization problem that minimizes the 'transport cost' between the two word distributions.

### Advantages of WMD:

*   **Semantic Understanding**: Captures semantic similarity, not just lexical overlap.
*   **Handles Synonymy**: Can find similarities between documents that use different words to express the same idea.

### Limitations:

*   **Computational Cost**: Can be computationally expensive, especially for long documents or large corpora, due to the EMD calculation.
*   **Requires Word Embeddings**: Relies on pre-trained word embeddings, and the quality of these embeddings directly impacts WMD's performance.

In [1]:
import gensim.downloader as api
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk

# Download NLTK data if not already present
# This needs to be run once
try:
    stopwords.words('english')
except LookupError:
    nltk.download('stopwords', quiet=True)
try:
    word_tokenize('test')
except LookupError:
    # It seems 'punkt' might not always cover 'punkt_tab' which is sometimes needed
    # for word_tokenize internally. Explicitly download both if needed.
    nltk.download('punkt', quiet=True)
    # Add an explicit download for 'punkt_tab' as suggested by the error
    nltk.download('punkt_tab', quiet=True)

# Load pre-trained Word2Vec model
# This might take a few minutes the first time
print("Loading Word2Vec model...")
model = api.load("word2vec-google-news-300")
print("Model loaded.")

def preprocess_text(text):
    # Tokenize words and remove stop words and non-alphabetic characters
    stop_words = set(stopwords.words('english'))
    word_tokens = word_tokenize(text.lower())
    filtered_sentence = [w for w in word_tokens if w.isalpha() and w not in stop_words]
    return filtered_sentence

# Example sentences
sentence_1 = "Obama speaks to the media in Illinois"
sentence_2 = "President greets the press in Chicago"
sentence_3 = "The cat sat on the mat"
sentence_4 = "A dog barks loudly"

# Preprocess sentences
processed_sentence_1 = preprocess_text(sentence_1)
processed_sentence_2 = preprocess_text(sentence_2)
processed_sentence_3 = preprocess_text(sentence_3)
processed_sentence_4 = preprocess_text(sentence_4)

# Calculate WMD between pairs of sentences
print("\nCalculating WMD scores...")
wmd_1_2 = model.wmdistance(processed_sentence_1, processed_sentence_2)
wmd_1_3 = model.wmdistance(processed_sentence_1, processed_sentence_3)
wmd_3_4 = model.wmdistance(processed_sentence_3, processed_sentence_4)

print(f"WMD between '{sentence_1}' and '{sentence_2}': {wmd_1_2:.4f}")
print(f"WMD between '{sentence_1}' and '{sentence_3}': {wmd_1_3:.4f}")
print(f"WMD between '{sentence_3}' and '{sentence_4}': {wmd_3_4:.4f}")

Loading Word2Vec model...
Model loaded.

Calculating WMD scores...
WMD between 'Obama speaks to the media in Illinois' and 'President greets the press in Chicago': 1.0175
WMD between 'Obama speaks to the media in Illinois' and 'The cat sat on the mat': 1.3346
WMD between 'The cat sat on the mat' and 'A dog barks loudly': 1.0761


In [3]:
# New example sentences
new_sentence_1 = "I love eating delicious pizza"
new_sentence_2 = "I enjoy consuming tasty pasta"

# Preprocess new sentences
processed_new_sentence_1 = preprocess_text(new_sentence_1)
processed_new_sentence_2 = preprocess_text(new_sentence_2)

# Calculate WMD for the new pair
wmd_new_pair = model.wmdistance(processed_new_sentence_1, processed_new_sentence_2)

print(f"WMD between '{new_sentence_1}' and '{new_sentence_2}': {wmd_new_pair:.4f}")

WMD between 'I love eating delicious pizza' and 'I enjoy consuming tasty pasta': 0.9054


In [5]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_sentence_vector(words, model):
    # Filter out words that are not in the model's vocabulary
    word_vectors = [model[word] for word in words if word in model.key_to_index]
    if not word_vectors:
        return np.zeros(model.vector_size) # Return a zero vector if no words are in vocabulary
    return np.mean(word_vectors, axis=0)

# Get sentence vectors for all example sentences
vec_1 = get_sentence_vector(processed_sentence_1, model)
vec_2 = get_sentence_vector(processed_sentence_2, model)
vec_3 = get_sentence_vector(processed_sentence_3, model)
vec_4 = get_sentence_vector(processed_sentence_4, model)

# Get sentence vectors for the new example sentences
vec_new_1 = get_sentence_vector(processed_new_sentence_1, model)
vec_new_2 = get_sentence_vector(processed_new_sentence_2, model)

# Calculate Cosine Similarity
# Cosine similarity expects 2D arrays, so reshape the vectors

print("\nCalculating Cosine Similarity scores...")

# Original pairs
cos_sim_1_2 = cosine_similarity(vec_1.reshape(1, -1), vec_2.reshape(1, -1))[0][0]
cos_sim_1_3 = cosine_similarity(vec_1.reshape(1, -1), vec_3.reshape(1, -1))[0][0]
cos_sim_3_4 = cosine_similarity(vec_3.reshape(1, -1), vec_4.reshape(1, -1))[0][0]

print(f"Cosine Similarity between '{sentence_1}' and '{sentence_2}': {cos_sim_1_2:.4f}")
print(f"Cosine Similarity between '{sentence_1}' and '{sentence_3}': {cos_sim_1_3:.4f}")
print(f"Cosine Similarity between '{sentence_3}' and '{sentence_4}': {cos_sim_3_4:.4f}")

# New pair
cos_sim_new_pair = cosine_similarity(vec_new_1.reshape(1, -1), vec_new_2.reshape(1, -1))[0][0]
print(f"Cosine Similarity between '{new_sentence_1}' and '{new_sentence_2}': {cos_sim_new_pair:.4f}")

print("\n--- Comparison --- ")
print(f"WMD between '{sentence_1}' and '{sentence_2}': {wmd_1_2:.4f}, Cosine Sim: {cos_sim_1_2:.4f}")
print(f"WMD between '{sentence_1}' and '{sentence_3}': {wmd_1_3:.4f}, Cosine Sim: {cos_sim_1_3:.4f}")
print(f"WMD between '{sentence_3}' and '{sentence_4}': {wmd_3_4:.4f}, Cosine Sim: {cos_sim_3_4:.4f}")
print(f"WMD between '{new_sentence_1}' and '{new_sentence_2}': {wmd_new_pair:.4f}, Cosine Sim: {cos_sim_new_pair:.4f}")


Calculating Cosine Similarity scores...
Cosine Similarity between 'Obama speaks to the media in Illinois' and 'President greets the press in Chicago': 0.6302
Cosine Similarity between 'Obama speaks to the media in Illinois' and 'The cat sat on the mat': 0.1175
Cosine Similarity between 'The cat sat on the mat' and 'A dog barks loudly': 0.4967
Cosine Similarity between 'I love eating delicious pizza' and 'I enjoy consuming tasty pasta': 0.7620

--- Comparison --- 
WMD between 'Obama speaks to the media in Illinois' and 'President greets the press in Chicago': 1.0175, Cosine Sim: 0.6302
WMD between 'Obama speaks to the media in Illinois' and 'The cat sat on the mat': 1.3346, Cosine Sim: 0.1175
WMD between 'The cat sat on the mat' and 'A dog barks loudly': 1.0761, Cosine Sim: 0.4967
WMD between 'I love eating delicious pizza' and 'I enjoy consuming tasty pasta': 0.9054, Cosine Sim: 0.7620
